# 🧠 Clinical Medical RAG for Epilepsy Research
### High-Performance Evidence-Grounded Retrieval-Augmented Generation System
**Stack:** `LangChain` | `FAISS` | `BAAI/bge-reranker-base` | `all-MiniLM-L6-v2` | `Ollama (Qwen 2.5:7B)` | `PyPDF`

---
### 📐 Architecture Overview
```
[User Clinical Query]
       │
       ▼
[Multi-Query Reformulation (Qwen 2.5:7B)] ──► 3 Focused Medical Search Variations
       │
       ▼
[FAISS Vector Database (MMR Search)] ──────► Diverse Candidate Chunks across Queries
       │
       ▼
[Global Deduplication] ─────────────────────► Unique Document Passages
       │
       ▼
[Cross-Encoder Deep Reranking (BGE)] ───────► Precision Relevance Scoring
       │
       ▼
[Strict Evidence-Grounded Prompting] ───────► Qwen 2.5:7B Clinical Synthesis with Citations
```

## 1. Environment Configuration & Cache Paths

In [ ]:
import os
import re
from typing import List, Dict, Any, Tuple
from dotenv import load_dotenv

# HuggingFace Local Storage Configuration
os.environ["HF_HOME"] = r"F:\HuggingFace"
os.environ["HF_HUB_CACHE"] = r"F:\HuggingFace\hub"
os.environ["TRANSFORMERS_CACHE"] = r"F:\HuggingFace\transformers"

load_dotenv()

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from sentence_transformers import CrossEncoder
from langchain_ollama import ChatOllama

print("✅ Frameworks and libraries imported successfully!")

## 2. Load Neural Embedding, Cross-Encoder & LLM Models

In [ ]:
# 1. Dense Embedding Model
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading Embedding Model: {embedding_model_name}...")
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

# 2. BGE Cross-Encoder Reranker
reranker_model_name = "BAAI/bge-reranker-base"
print(f"Loading Cross-Encoder Reranker: {reranker_model_name}...")
reranker = CrossEncoder(reranker_model_name)

# 3. Local Ollama LLM (Qwen 2.5:7B)
print("Connecting to local Ollama Qwen 2.5:7B...")
llm = ChatOllama(model="qwen2.5:7b", temperature=0.1)

print("\n🚀 All neural models loaded successfully!")

## 3. PDF Ingestion & Structure-Aware Chunking

In [ ]:
pdf_path = r"F:\Lectures\AI Hacathon\epilepsy.pdf"
faiss_dir = r"F:\Lectures\AI Hacathon\FAISS_DB_V2"

print(f"Parsing PDF: {pdf_path}")
loader = PyPDFLoader(pdf_path)
documents = loader.load()
print(f"Loaded {len(documents)} pages from PDF.")

# Chunking optimized to preserve tables and complete medical associations
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=250,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} text chunks.")

# Build and save FAISS Vector Database
vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)
vectorstore.save_local(faiss_dir)
print(f"FAISS Vector Database saved successfully at: {faiss_dir}")

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 8, "fetch_k": 30, "lambda_mult": 0.6}
)

## 4. Multi-Query Expansion, Retrieval, Reranking & Grounded Synthesis Pipeline

In [ ]:
def ask_clinical_rag(query: str, top_k: int = 8) -> Dict[str, Any]:
    print("=" * 80)
    print(f"[USER QUESTION]: {query}")
    print("=" * 80)

    # Step 1: Multi-Query Expansion
    expansion_prompt = f"""You are a specialized medical query expansion assistant.
Given the clinical research question below, generate exactly 3 short, specific search queries covering different aspects, clinical terms, synonyms, and specific therapies mentioned.

Question: {query}

Rules:
- Return ONLY the 3 queries, one per line.
- Do NOT include numbering, bullet points, or preamble.
"""
    resp = llm.invoke(expansion_prompt)
    lines = resp.content.strip().split("\n")
    expanded = []
    for l in lines:
        cleaned = re.sub(r"^[\d\.\-\s\*\)]+", "", l.strip())
        if cleaned and len(cleaned) > 5 and not cleaned.lower().startswith("here are"):
            expanded.append(cleaned)
    
    retrieval_queries = [query] + expanded[:3]
    print("\n🔍 Generated Retrieval Queries:")
    for idx, q in enumerate(retrieval_queries, start=1):
        print(f"  {idx}. {q}")

    # Step 2: Diverse MMR Retrieval
    all_candidates = []
    for q in retrieval_queries:
        docs = retriever.invoke(q)
        all_candidates.extend(docs)

    # Step 3: Global Deduplication
    unique_docs = {}
    for d in all_candidates:
        txt = d.page_content.strip()
        if txt not in unique_docs:
            unique_docs[txt] = d
    unique_candidates = list(unique_docs.values())
    print(f"\nRetrieved {len(all_candidates)} candidates across queries -> {len(unique_candidates)} unique chunks.")

    # Step 4: Cross-Encoder Reranking
    pairs = [[query, doc.page_content] for doc in unique_candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(unique_candidates, [float(s) for s in scores]), key=lambda x: x[1], reverse=True)
    top_docs = ranked[:top_k]

    print("\n📊 Top Retrieved Clinical Evidence Chunks:")
    for i, (doc, score) in enumerate(top_docs[:3], start=1):
        page = doc.metadata.get("page", 0) + 1
        print(f"  [Source {i} | Page {page} | Score: {score:.4f}] {doc.page_content[:140]}...")

    # Step 5: Evidence Context Construction
    context_parts = []
    for i, (doc, score) in enumerate(top_docs, start=1):
        page = doc.metadata.get("page", 0) + 1
        context_parts.append(f"[Source {i} | Page {page} | Relevance Score: {score:.4f}]\n{doc.page_content.strip()}")
    context_text = "\n\n" + ("=" * 50) + "\n\n".join(context_parts)

    # Step 6: Grounded Medical Response Generation
    prompt = f"""You are an expert clinical neurology research assistant specializing in epilepsy and neurotherapeutics.

Your task is to provide a complete, accurate, and rigorous answer to the user's clinical question based EXCLUSIVELY on the provided document excerpts.

CRITICAL GUIDELINES:
1. Ground every claim directly in the provided context.
2. If multiple therapies, devices, or percentages are requested, enumerate each one clearly with bullet points.
3. Preserve all numerical percentages, trial names, and responder rates exactly as reported.
4. Attribute each responder rate or outcome specifically to its corresponding device/treatment.
5. Provide a concise synthesis followed by structured details.

DOCUMENT CONTEXT:
{context_text}

USER CLINICAL QUESTION:
{query}

CLINICAL ANSWER:"""

    response = llm.invoke(prompt)
    answer = response.content.strip()

    print("\n" + "=" * 80)
    print("FINAL EVIDENCE-GROUNDED ANSWER:")
    print("=" * 80)
    print(answer)

    return {
        "answer": answer,
        "sources": top_docs,
        "queries": retrieval_queries
    }

## 5. Benchmark Query Evaluation

In [ ]:
query = "What are the three invasive neuromodulation options approved for adult drug-resistant epilepsy, and what are their typical responder rates?"
result = ask_clinical_rag(query, top_k=8)